# Shortest Path by Combinatorics — Try Every Path, Keep the Cheapest

The clever way to find a shortest path is Dijkstra's algorithm. The **honest,
brute-force** way is combinatorial: *enumerate the possible paths from source to
destination, add up each one's edge weights, and keep the minimum.*

This notebook builds the brute-force finder two ways, so the combinatorics is
visible:

1. **The permutations view** — literally try every ordering of the intermediate
   nodes (`itertools.permutations`). This makes the *factorial* cost obvious.
2. **Backtracking enumeration** — walk the graph and list every *simple* path
   (no repeated vertices). Smarter: it never builds orderings that aren't real
   paths.

Then we **verify** the answer against Dijkstra, and look at *why* the
combinatorial approach explodes on bigger graphs.

Pure Python standard library — no installs.

## 1. The graph

A weighted **directed** graph as an adjacency dict: `graph[u][v] = weight of edge
u→v`. We'll find the cheapest route from `A` to `E`.

In [ ]:
GRAPH = {
    "A": {"B": 1, "C": 4, "D": 7},
    "B": {"C": 1, "D": 5},
    "C": {"D": 1, "E": 3},
    "D": {"E": 1},
    "E": {},
}
SRC, DST = "A", "E"

def path_cost(graph, path):
    """Total weight of a path (list of nodes), or None if any edge is missing."""
    total = 0
    for u, v in zip(path, path[1:]):
        if v not in graph[u]:
            return None            # this ordering isn't a real path
        total += graph[u][v]
    return total

print("A\u2192B\u2192C\u2192D\u2192E costs:", path_cost(GRAPH, ["A", "B", "C", "D", "E"]))
print("A\u2192D\u2192B  (no edge D\u2192B) \u2192", path_cost(GRAPH, ["A", "D", "B"]))

## 2. The permutations view — where the combinatorics shows up

A path from `A` to `E` is `A`, then some ordered selection of the *other* nodes,
then `E`. So we can generate candidates by taking every ordering of every subset
of the intermediate nodes `{B, C, D}`, sandwiching it between `A` and `E`, and
keeping the ones that happen to be valid paths.

The number of candidate orderings is $\sum_{k=0}^{m} \frac{m!}{(m-k)!}$ where $m$
is the count of intermediate nodes — this is what makes brute force expensive.

In [ ]:
import itertools

def shortest_path_permutations(graph, src, dst):
    intermediate = [n for n in graph if n not in (src, dst)]
    candidates_tried = 0
    valid = []
    # every ordering of every subset of the intermediate nodes
    for k in range(len(intermediate) + 1):
        for perm in itertools.permutations(intermediate, k):
            candidates_tried += 1
            path = [src, *perm, dst]
            cost = path_cost(graph, path)
            if cost is not None:
                valid.append((cost, path))
    best = min(valid) if valid else None
    return best, valid, candidates_tried

best, valid, tried = shortest_path_permutations(GRAPH, SRC, DST)
print(f"orderings tried: {tried}   valid paths: {len(valid)}")
for cost, path in sorted(valid):
    print(f"  cost {cost}:  {' \u2192 '.join(path)}")
print(f"\nSHORTEST: {' \u2192 '.join(best[1])}  (cost {best[0]})")

## 3. Backtracking enumeration — list every simple path

The permutations view wastes work: most orderings aren't real paths (no edge
exists), and it can't handle graphs where a node repeats. **Backtracking** walks
outward from the source along real edges only, and records a path whenever it
reaches the destination. `visited` prevents cycles, so these are *simple* paths.

In [ ]:
def all_simple_paths(graph, src, dst):
    """Every simple path src→dst, found by depth-first backtracking."""
    paths = []
    def dfs(node, visited, acc):
        if node == dst:
            paths.append(list(acc))
            return
        for nbr in graph[node]:            # only real edges
            if nbr not in visited:
                visited.add(nbr); acc.append(nbr)
                dfs(nbr, visited, acc)
                acc.pop(); visited.remove(nbr)   # backtrack
    dfs(src, {src}, [src])
    return paths

def shortest_path_backtracking(graph, src, dst):
    paths = all_simple_paths(graph, src, dst)
    if not paths:
        return None, []
    scored = sorted((path_cost(graph, p), p) for p in paths)
    return scored[0], scored

best, scored = shortest_path_backtracking(GRAPH, SRC, DST)
print(f"simple paths found: {len(scored)}")
for cost, path in scored:
    print(f"  cost {cost}:  {' \u2192 '.join(path)}")
print(f"\nSHORTEST: {' \u2192 '.join(best[1])}  (cost {best[0]})")

## 4. Verify against Dijkstra

Brute force is easy to get subtly wrong, so we check it against the standard
algorithm. They must agree on the cost (multiple paths can tie on cost).

In [ ]:
import heapq

def dijkstra(graph, src, dst):
    pq = [(0, src, [src])]            # (cost so far, node, path)
    best_cost = {src: 0}
    while pq:
        cost, node, path = heapq.heappop(pq)
        if node == dst:
            return cost, path
        for nbr, w in graph[node].items():
            nc = cost + w
            if nc < best_cost.get(nbr, float("inf")):
                best_cost[nbr] = nc
                heapq.heappush(pq, (nc, nbr, path + [nbr]))
    return None

dij_cost, dij_path = dijkstra(GRAPH, SRC, DST)
brute_cost = best[0]
print(f"Dijkstra   : {' \u2192 '.join(dij_path)}  (cost {dij_cost})")
print(f"Brute force: {' \u2192 '.join(best[1])}  (cost {brute_cost})")
assert dij_cost == brute_cost, "mismatch!"
print("\n\u2713 costs agree")

## 5. Why combinatorial search explodes

On a **complete** graph of `n` nodes, the number of simple paths between two
fixed nodes is $\sum_{k=0}^{n-2}\frac{(n-2)!}{(n-2-k)!}$ — it grows faster than
$(n-2)!$. Dijkstra, by contrast, is $O(E \log V)$. Let's watch the path count
blow up.

In [ ]:
def complete_graph(n):
    nodes = list(range(n))
    return {u: {v: 1 for v in nodes if v != u} for u in nodes}

print(f"{'n':>3} | {'simple paths 0\u2192n-1':>20} | {'time (s)':>9}")
print("-" * 40)
import time
for n in range(3, 10):
    g = complete_graph(n)
    t0 = time.perf_counter()
    paths = all_simple_paths(g, 0, n - 1)
    dt = time.perf_counter() - t0
    print(f"{n:>3} | {len(paths):>20,} | {dt:>9.4f}")

## 6. Putting it together — one function

A single entry point that returns the shortest path, its cost, and how many
candidate paths it examined (so you can *see* the combinatorial work).

In [ ]:
def shortest_path(graph, src, dst):
    """Combinatorial shortest path: enumerate simple paths, return the cheapest.
    Returns (path, cost, num_paths_examined) or (None, None, 0) if unreachable."""
    paths = all_simple_paths(graph, src, dst)
    if not paths:
        return None, None, 0
    cost, path = min((path_cost(graph, p), p) for p in paths)
    return path, cost, len(paths)

path, cost, examined = shortest_path(GRAPH, "A", "E")
print(f"shortest : {' \u2192 '.join(path)}")
print(f"cost     : {cost}")
print(f"examined : {examined} candidate paths")

# unreachable example
print("\nA\u2192A? ", shortest_path(GRAPH, "A", "A"))       # trivial: [A], 0
print("E\u2192A? ", shortest_path(GRAPH, "E", "A"))       # no outgoing edges from E

## 7. Takeaways

- **Combinatorial (brute-force) shortest path** = enumerate paths, score, pick the
  min. Simple, always correct, and it naturally hands you *every* alternative
  route — useful when you want the top-k paths, not just the best one.
- It's **exponential**: fine for small graphs (≤ ~10 nodes) and for teaching, but
  Dijkstra ($O(E \log V)$) is the tool for anything real.
- **Backtracking beats the permutations view** — it only ever extends along real
  edges, so it wastes no work on orderings that aren't paths, and it generalizes
  to graphs the permutation sandwich can't express.

### Extensions
- Return the **k shortest paths** (`sorted(scored)[:k]`) — trivial here, awkward
  with plain Dijkstra.
- Add a **max-length / max-hops cap** to prune the search early.
- Try it on an **undirected** graph (make edges symmetric) or one with **cycles**
  — the `visited` set already handles cycles correctly.